In [92]:
import os
import datetime
import logging
import pandas as pd
from pathlib import Path
from delta.tables import DeltaTable
from pyspark.sql import SparkSession, functions as F
import pandas as pd


from helpers_spark.serialization import deserialize_json_columns, deserialize_json_columns


# from ingest_helpers.spark_processing_helpers2 import build_tree, normalize_tree, sort_tree, select_tree,  explode_nested, split_nested

# from ingest_helpers.spark_processing_helpers2_test import apply_enrich

# -----------------------------
# Logging setup
# -----------------------------
LOG_DIR = Path(os.getenv("LOG_DIR", "/logs"))
LOG_DIR.mkdir(parents=True, exist_ok=True)
LOG_FILE = LOG_DIR / "discogs_silver_ingest.log"

logger = logging.getLogger(__name__)
if not logger.handlers:
    logging.basicConfig(
        level=logging.INFO,
        format="%(asctime)s | %(levelname)-8s | %(name)s | %(message)s",
        handlers=[logging.StreamHandler(), logging.FileHandler(LOG_FILE)],
    )

logger.info(f"Logging to {LOG_FILE}")

# -----------------------------
# Environment / config
# -----------------------------
DATA_DIR = os.getenv("DATA_DIR")
BRONZE_DATA_DIR_RELEASES = os.path.join(DATA_DIR, "bronze","releases")

# -----------------------------
# Spark session
# -----------------------------
spark = (
    SparkSession.builder.appName("discogs-silver-ingest")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config(
        "spark.sql.catalog.spark_catalog",
        "org.apache.spark.sql.delta.catalog.DeltaCatalog",
    )
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
spark.conf.set("spark.databricks.delta.schema.autoMerge.enabled", "true")



logger.info(f"loading {BRONZE_DATA_DIR}")        
df = spark.read.format("delta").load(BRONZE_DATA_DIR_RELEASES)

2026-02-25 12:34:56,494 | INFO     | __main__ | Logging to /logs/discogs_silver_ingest.log
2026-02-25 12:34:56,496 | INFO     | __main__ | loading /data_tests/bronze/artists


In [90]:
df.columns


['@id',
 '@status',
 'artists',
 'companies',
 'country',
 'data_quality',
 'extraartists',
 'formats',
 'genres',
 'identifiers',
 'labels',
 'master_id',
 'notes',
 'released',
 'series',
 'styles',
 'title',
 'tracklist',
 'videos',
 'root_hash',
 'last_dump_update',
 'artists_hash',
 'extraartists_hash',
 'tracklist_hash']

In [91]:
df.select("@id", "title", "released","country","last_dump_update").show()

+----+--------------------+----------+-------+----------------+
| @id|               title|  released|country|last_dump_update|
+----+--------------------+----------+-------+----------------+
|3114|                Slip|2000-12-00|     US|        20251101|
|2235|Funkin With The D...|1985-00-00|     US|        20251101|
|1503|Groove Now / New ...|1997-07-00|     UK|        20251101|
|2990|           Reference|2000-06-26|     UK|        20251101|
|2741|            Gorgeous|1998-06-00|     UK|        20251101|
|1873|       Blue Funk III|      1995|     US|        20251101|
| 913|               Flava|      1997|Germany|        20251101|
|2476|           Speed 127|      1990|Belgium|        20251101|
|1067|     The Head Hunter|1991-09-00|     UK|        20251101|
|3482|Decadence / Kreva...|2001-03-19|     UK|        20251101|
|2272|Love Baby / Midni...|1990-11-00|     UK|        20251101|
|4126|         Peristaltik|2001-00-00|Germany|        20251101|
|4310|Wonderful Person ...|      1999|  

In [85]:
df.schema

StructType([StructField('@id', LongType(), True), StructField('@status', StringType(), True), StructField('artists', StringType(), True), StructField('companies', StringType(), True), StructField('country', StringType(), True), StructField('data_quality', StringType(), True), StructField('extraartists', StringType(), True), StructField('formats', StringType(), True), StructField('genres', StringType(), True), StructField('identifiers', StringType(), True), StructField('labels', StringType(), True), StructField('master_id', StringType(), True), StructField('notes', StringType(), True), StructField('released', StringType(), True), StructField('series', StringType(), True), StructField('styles', StringType(), True), StructField('title', StringType(), True), StructField('tracklist', StringType(), True), StructField('videos', StringType(), True), StructField('complete_root_hash', StringType(), True), StructField('last_dump_update', StringType(), True)])

In [65]:
df.select("groups").show()

+--------------------+
|              groups|
+--------------------+
|                NULL|
|{"name":[{"@id":1...|
|                NULL|
|                NULL|
|                NULL|
|                NULL|
|{"name":[{"@id":1...|
|                NULL|
|                NULL|
|                NULL|
|                NULL|
|                NULL|
|                NULL|
|                NULL|
|                NULL|
|                NULL|
|                NULL|
|                NULL|
|                NULL|
|                NULL|
+--------------------+
only showing top 20 rows


In [75]:
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, ArrayType, StringType, LongType

groups_schema = StructType([
    StructField("name", ArrayType(
        StructType([
            StructField("@id", LongType()),
            StructField("_text", StringType())
        ])
    ))
])

df_groups = (
    df
    .withColumn("groups_struct", F.from_json("groups", groups_schema))
    .withColumn("group", F.explode_outer("groups_struct.name"))
    .select(
        F.col("id").alias("artist_id"),
        F.col("name").alias("artist_name"),
        F.col("group.@id").alias("group_id"),
        F.col("group._text").alias("group_name"),
        "last_dump_update"
    )
)

df_groups.show()

+---------+-------------+--------+--------------------+----------------+
|artist_id|  artist_name|group_id|          group_name|last_dump_update|
+---------+-------------+--------+--------------------+----------------+
|        1|The Persuader|    NULL|                NULL|        20251101|
|        5|   Heiko Laux|    1320|           Orange 25|        20251101|
|        5|   Heiko Laux|    6874|Total Planet Refr...|        20251101|
|        5|   Heiko Laux|   10281|            Item One|        20251101|
|        5|   Heiko Laux|   20864|               RezzQ|        20251101|
|        5|   Heiko Laux|   22058|        Monobleeding|        20251101|
|        5|   Heiko Laux|   22069|       Laux & Olsson|        20251101|
|        5|   Heiko Laux|   24709|Direct From The M...|        20251101|
|        5|   Heiko Laux|   27085|              Sodiac|        20251101|
|        5|   Heiko Laux|   83311| The Global Segments|        20251101|
|        5|   Heiko Laux|  266516|       Offshore F

In [53]:
df.printSchema()


root
 |-- aliases: string (nullable = true)
 |-- data_quality: string (nullable = true)
 |-- groups: string (nullable = true)
 |-- id: long (nullable = true)
 |-- members: string (nullable = true)
 |-- name: string (nullable = true)
 |-- namevariations: string (nullable = true)
 |-- profile: string (nullable = true)
 |-- realname: string (nullable = true)
 |-- urls: string (nullable = true)
 |-- complete_root_hash: string (nullable = true)
 |-- last_dump_update: string (nullable = true)



In [63]:
df_artists = (df.select(
    "@id",
    explode("artists.artist").alias("artist")
)
    .select("@id", "artist.id", "artist.name")
    .withColumn("track_position",F.lit("All"))
    .withColumn("role",F.lit("Release Artist"))
    .withColumnsRenamed({"id":"artist_id", "name":"artist_name"})
    .withColumnRenamed("@id","release_id")
             )

df_artists.show()

+----------+---------+--------------------+--------------+--------------+
|release_id|artist_id|         artist_name|track_position|          role|
+----------+---------+--------------------+--------------+--------------+
|         1|        1|       The Persuader|           All|Release Artist|
|         2|        2|Mr. James Barth &...|           All|Release Artist|
|         3|        3|           Josh Wink|           All|Release Artist|
|         4|       21|         Faze Action|           All|Release Artist|
|         5|       22|            Datacide|           All|Release Artist|
|         6|        2|Mr. James Barth &...|           All|Release Artist|
|         7|       28|        Moonchildren|           All|Release Artist|
|         8|       29|       Sweet Abraham|           All|Release Artist|
|         9|       33|            Blue Six|           All|Release Artist|
|        10|       36|          Lovetronic|           All|Release Artist|
|        11|       33|            Blue

In [163]:
import pyspark.sql.functions as F

df_extraartists = (
    df
    .select(
        F.col("@id").alias("release_id"),
        F.explode("extraartists.artist").alias("artist")
    )
    .select(
        "release_id",
        F.col("artist.id").alias("artist_id"),
        F.col("artist.name").alias("artist_name"),
        F.col("artist.role"),
        F.col("artist.tracks").alias("track_position")
    ).withColumn(
        "track_position",
        F.when(
            F.col("track_position").isNull(),
            F.lit("All")
        ).otherwise(F.col("track_position"))
    )
    .withColumn("role", F.explode(F.split("role", ", ")))
    .withColumn(
        "track_position",
        F.explode(F.split("track_position", ", "))
    )
    .withColumn("track_title", F.lit("All"))
)

df_extraartists.show()


+----------+---------+--------------------+--------------------+--------------+-----------+
|release_id|artist_id|         artist_name|                role|track_position|track_title|
+----------+---------+--------------------+--------------------+--------------+-----------+
|         1|   507025|George Cutmaster ...|      Lacquer Cut By|           All|        All|
|         1|      239|     Jesper Dahlbäck|Written-By [All T...|           All|        All|
|         2|       26|        Alexi Delano|            Producer|           All|        All|
|         2|       26|        Alexi Delano|         Recorded By|           All|        All|
|         2|       27|      Cari Lekebusch|            Producer|           All|        All|
|         2|       27|      Cari Lekebusch|         Recorded By|           All|        All|
|         2|       26|        Alexi Delano|          Written-By|           All|        All|
|         2|       27|      Cari Lekebusch|          Written-By|           All| 

In [123]:
from pyspark.sql.functions import split


df_track_artists = (df.select(
    "@id",

    explode("tracklist.track").alias("tracks"))
        .select("@id", "tracks.*")
       

)
df_track_artists.show()

+---+--------------------+--------+--------------------+--------+--------------------+
|@id|             artists|duration|        extraartists|position|               title|
+---+--------------------+--------+--------------------+--------+--------------------+
|  1|                NULL|    4:45|                NULL|       A|           Östermalm|
|  1|                NULL|    6:11|                NULL|      B1|          Vasastaden|
|  1|                NULL|    2:49|                NULL|      B2|         Kungsholmen|
|  1|                NULL|    5:38|                NULL|      C1|           Södermalm|
|  1|                NULL|    4:52|                NULL|      C2|            Norrmalm|
|  1|                NULL|    5:16|                NULL|       D|          Gamla Stan|
|  2|                NULL|    5:08|                NULL|      A1|         A Sea Apart|
|  2|                NULL|    4:21|                NULL|      A2|         Dutchmaster|
|  2|                NULL|    4:22|        

In [91]:

df_tracks_artists = (df.select(
    "@id",

    explode("tracklist.track")
        .alias("tracks"))
        .select("@id", explode("tracks.artists.artist")
            .alias("artists"),"tracks.position","tracks.title")
    .select("@id","position","title","artists.id","artists.name")
    .withColumn("role",F.lit("Track artist"))
    .withColumnsRenamed({"id":"artist_id","title":"track_title","position":"track_position", "name":"artist_name"})
    .withColumnRenamed("@id","release_id")
    
       

)
df_tracks_artists.show()

+----------+--------------+--------------------+---------+--------------------+------------+
|release_id|track_position|         track_title|artist_id|         artist_name|        role|
+----------+--------------+--------------------+---------+--------------------+------------+
|         3|             1|                  D2|        4|       Johannes Heil|Track artist|
|         3|             1|                  D2|        5|          Heiko Laux|Track artist|
|         3|             2|    Anjua (Sneaky 3)|    15525|   Karl Axel Bissler|Track artist|
|         3|             3|When The Funk Hit...|        7|            Sylk 130|Track artist|
|         3|             4|What's The Time, ...|        1|       The Persuader|Track artist|
|         3|             5|              Vol. 2|   267132|    Care Company (2)|Track artist|
|         3|             6|  Political Prisoner|     6981|          Gez Varley|Track artist|
|         3|             7|         Pop Kulture|       11|            

In [161]:

df_tracks_extraartists = (df.select(
    "@id",

    explode("tracklist.track")
        .alias("tracks"))
        .select("@id", explode("tracks.extraartists.artist")
            .alias("artists"),"tracks.position","tracks.title")
    .select("@id","position","title","artists.id","artists.name","artists.role")

    
    .withColumnsRenamed({"id":"artist_id","title":"track_title","position":"track_position", "name":"artist_name"})
    .withColumnRenamed("@id","release_id").select(
    "release_id",
    "artist_id",
    "artist_name",
    "track_position",
    "track_title",
    explode(split("role", ", ")).alias("role"))
                  )


df_tracks_extraartists.show()

+----------+---------+------------------+--------------+--------------------+--------------+
|release_id|artist_id|       artist_name|track_position|         track_title|          role|
+----------+---------+------------------+--------------+--------------------+--------------+
|         3|        8|     Mood II Swing|             3|When The Funk Hit...|         Remix|
|         3|       23|        Alex Hi-Fi|             8|K-Mart Shopping (...|         Remix|
|         3|       14|  Eight Miles High|             9|Lovelee Dae (Eigh...|         Remix|
|         3|    67226|     Stacey Pullen|            10|               Sweat|     Presenter|
|         4|   246316|         Raj Gupta|             2|  To Love Is To Grow|    Written-By|
|         4|    56668|      Zeke Manyika|             2|  To Love Is To Grow|    Written-By|
|         4|     9559|   Vanessa Freeman|             4|           Heartbeat|    Written-By|
|         4|    56668|      Zeke Manyika|             6|   Got To Find

In [156]:
df_artists_all =( 
    df_tracks_extraartists
    .unionByName(df_tracks_artists, allowMissingColumns=True)    
        .unionByName(df_extraartists, allowMissingColumns=True)    
        .unionByName(df_artists, allowMissingColumns=True).
        distinct()
                )

df_artists_all.show()

+----------+---------+--------------------+--------------+--------------------+--------------------+
|release_id|artist_id|         artist_name|track_position|         track_title|                role|
+----------+---------+--------------------+--------------+--------------------+--------------------+
|        37|    12433|Love From San Fra...|             5|165 Drop (Love Fr...|               Remix|
|       102|  1147165|   Randy Johnson (3)|            A2|Why U Wanna? (Ale...|              Vocals|
|       128|   476951|       Pete Williams|            A2|Jive (Natural Rhy...|            Music By|
|       237|   168579|            Nico (4)|            A1|        Killamanjaro|            Mixed By|
|       251|      278|              Peshay|             A|Music (Peshay Rew...|      Remix [Rework]|
|       297|      391|          David Toop|             2|Let All Mortal Fl...|         Arranged By|
|       320|      466|                Plug|           1-1|         Lunar Laugh|            